<a href="https://colab.research.google.com/github/contreras-juan/Material-Ciencia-de-Datos/blob/main/Machine_Learning/Ejercicios/01_Taller_Regresion_Clasificacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Taller práctico: Regresión y clasificación</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Instrucciones</h2>

Este taller integra los temas de los cuadernos de **regresión lineal** (01–03), **regresión logística** (04),
**matriz de confusión** (05), **SVM** (06), **KNN** (07), **árboles de decisión** (08) y **validación cruzada** (10),
pero con **datos públicos distintos** a los de las notas (allí usamos California Housing, Diabetes de sklearn, Iris y Cáncer de mama):

| Parte | Problema | Dataset | Variable objetivo |
|---|---|---|---|
| I | Regresión | [Medical Cost Personal](https://github.com/stedy/Machine-Learning-with-R-datasets) (1 338 asegurados en EE. UU.) | `charges`: costo médico anual facturado (USD) |
| II | Clasificación binaria | [Pima Indians Diabetes](https://github.com/jbrownlee/Datasets) (768 mujeres pima, NIDDK) | `diabetes`: 1 si desarrolló diabetes en 5 años |

En cada parte recorrerás el mismo ciclo: **explorar → preparar → entrenar → evaluar → interpretar**.

**Cómo trabajar:**

- Completa las celdas marcadas con `# Escribe tu código aquí`.
- Responde las preguntas en una celda Markdown debajo de tu código. Las preguntas de interpretación valen tanto como el código.
- Los escaladores se ajustan **solo con entrenamiento** (`fit_transform` en entrenamiento y `transform` en prueba, como en el cuaderno 03). Para no olvidarlo, une escalador y modelo con `make_pipeline` (cuaderno 01).
- Los hiperparámetros se eligen con **validación cruzada sobre entrenamiento** (`GridSearchCV`, `cross_val_score`, cuaderno 10). El conjunto de prueba se usa **una sola vez**, al final, para reportar.
- Usa `SEED = 42` en todo lo que tenga aleatoriedad para que tus resultados sean comparables con los de tus compañeros.

<h2 style="color: #007ACC;">Tabla de contenido</h2>

**Parte I — Regresión: costos médicos**

- [Ejercicio 1. Exploración y partición](#ejercicio-1)
- [Ejercicio 2. Regresión lineal e interpretación de coeficientes](#ejercicio-2)
- [Ejercicio 3. Regularización: Ridge y Lasso](#ejercicio-3)
- [Ejercicio 4. KNN, SVR y árbol de regresión](#ejercicio-4)
- [Ejercicio 5. Interpretación de modelos de regresión](#ejercicio-5)

**Parte II — Clasificación: diabetes en mujeres pima**

- [Ejercicio 6. Exploración y limpieza](#ejercicio-6)
- [Ejercicio 7. Regresión logística y matriz de confusión](#ejercicio-7)
- [Ejercicio 8. El umbral de decisión](#ejercicio-8)
- [Ejercicio 9. KNN, SVM y árbol de clasificación](#ejercicio-9)
- [Ejercicio 10. Comparación con validación cruzada](#ejercicio-10)
- [Ejercicio 11. Interpretación de clasificadores](#ejercicio-11)

**Cierre**

- [Ejercicio integrador. Informe para el comité](#ejercicio-integrador)

---

<h2 style="color: #007ACC;">Instalación e importaciones</h2>

Ejecuta primero la celda de instalación y luego la de importaciones.

In [ ]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib seaborn scikit-learn statsmodels

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.3f}".format)

SEED = 42
URL_SEGUROS = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
URL_PIMA = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

Funciones auxiliares para reportar las métricas de los cuadernos 01 y 04 de forma homogénea. **No las modifiques**; úsalas en los ejercicios.

In [ ]:
def metricas_regresion(y_true, y_pred):
    """MAE, RMSE (en las unidades de y) y R² (cuaderno 01)."""
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": r2_score(y_true, y_pred),
    }


def metricas_clasificacion(y_true, y_pred):
    """Exactitud, precisión, sensibilidad y F1 de la clase positiva (cuaderno 04)."""
    return {
        "exactitud": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensibilidad": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
    }

---

<h1 style="color: #FECB05;">Parte I — Regresión: costos médicos</h1>

Una aseguradora quiere estimar el **costo médico anual** (`charges`) de un asegurado a partir de:

| Variable | Descripción |
|---|---|
| `age` | Edad del asegurado principal |
| `sex` | `female` / `male` |
| `bmi` | Índice de masa corporal (kg/m²). Obesidad: $\text{IMC} \geq 30$ |
| `children` | Número de hijos cubiertos |
| `smoker` | `yes` / `no` |
| `region` | Región de residencia en EE. UU. (`northeast`, `northwest`, `southeast`, `southwest`) |

Además de predecir bien, el área de suscripción necesita **entender** qué factores mueven el costo, así que la interpretación es parte del entregable.

---

<a id="ejercicio-1"></a>
<h2 style="color: #007ACC;">Ejercicio 1. Exploración y partición</h2>

**Objetivo:** conocer la variable objetivo, convertir las categóricas en variables dicotómicas y dejar lista la partición.

1. Carga los datos desde `URL_SEGUROS` en un DataFrame `seguros`. Revisa dimensiones, tipos, faltantes y estadísticas descriptivas.
2. Grafica en una sola figura: (a) histograma de `charges`, (b) histograma de $\ln(\text{charges})$ y (c) dispersión `bmi` vs `charges` coloreada por `smoker`.
3. Calcula la media y la mediana de `charges` por `smoker`, y muestra el mapa de calor de correlaciones de las variables numéricas.
4. Convierte `sex`, `smoker` y `region` en variables dicotómicas con `pd.get_dummies(..., drop_first=True, dtype=int)` (cuaderno 08) y guarda el resultado en `seguros_dum`.
5. Separa `X_reg` (todas las columnas menos `charges`) y `y_reg`, y haz una partición 80/20 con `random_state=SEED`: `X_tr_r, X_te_r, y_tr_r, y_te_r`.
6. **Preguntas:**
   - ¿Cómo es la distribución de `charges`? ¿Qué implica para el MAE frente al RMSE?
   - Describe los "grupos" que ves en la dispersión `bmi` vs `charges`. ¿Qué variable los separa y qué sugiere para un modelo lineal?
   - Las tres columnas de región (`northwest`, `southeast`, `southwest`) más `northeast` siempre suman 1. ¿Por qué eso sería un problema de **multicolinealidad** (cuaderno 03) en un modelo con intercepto? ¿Cómo lo resuelve `drop_first=True`?

In [ ]:
# Escribe tu código aquí
seguros = pd.read_csv(URL_SEGUROS)

---

<a id="ejercicio-2"></a>
<h2 style="color: #007ACC;">Ejercicio 2. Regresión lineal e interpretación de coeficientes</h2>

**Objetivo:** entrenar el modelo lineal, compararlo contra una referencia sencilla y leer sus coeficientes e inferencia (cuadernos 01 y 02).

1. **Referencia.** Predice para todo el conjunto de prueba la **media de `y_tr_r`** y calcula `metricas_regresion`. Relaciona el resultado con la fórmula del $R^2 = 1 - SS_{res}/SS_{tot}$: ¿qué modelo representa $SS_{tot}$?
2. Entrena `reg_lin = LinearRegression()` con `X_tr_r` **sin escalar** y reporta las métricas en entrenamiento y prueba.
3. Construye una tabla con el coeficiente de cada variable. Como no escalamos, cada $\beta_j$ se lee como "cambio en USD de `charges` cuando $x_j$ aumenta en una unidad, manteniendo lo demás constante". Redacta una frase para `age`, `bmi` y `smoker_yes`.
4. Ajusta el mismo modelo con `statsmodels` (`sm.OLS(y_tr_r, sm.add_constant(X_tr_r)).fit()`, cuaderno 02). ¿Coinciden los coeficientes? ¿Qué variables **no** son significativas al 5 %? Interpreta el intervalo de confianza de `smoker_yes`.
5. Grafica residuales ($y - \hat{y}$) vs. predicciones en prueba, coloreando por `smoker_yes`. Revisa los supuestos de **linealidad** y **homocedasticidad** (cuaderno 01).
6. **Transformaciones de variables.** El cuaderno 01 muestra que un modelo con $x^2$ o $\ln(x)$ sigue siendo lineal en los parámetros. Escribe `agregar_variables(df)` que añada:
   - `age2 = age**2` (término cuadrático),
   - `obeso = 1` si `bmi >= 30`,
   - `fumador_obeso = obeso * smoker_yes` (vale 1 solo para fumadores obesos).

   Reentrena el modelo lineal con estas variables (`reg_lin2`), compara las métricas de prueba y estima de nuevo con `statsmodels`. Interpreta el coeficiente de `fumador_obeso`.

In [ ]:
# Escribe tu código aquí

---

<a id="ejercicio-3"></a>
<h2 style="color: #007ACC;">Ejercicio 3. Regularización: Ridge y Lasso</h2>

**Objetivo:** elegir la fuerza de regularización por validación cruzada y usar Lasso para ver qué variables pesan más (cuadernos 01 y 03).

Trabaja con las variables transformadas (`X_tr_r2`).

1. Define `cv_reg = KFold(n_splits=5, shuffle=True, random_state=SEED)` (cuaderno 10).
2. Con `GridSearchCV` (`cv=cv_reg`, `scoring="neg_mean_absolute_error"`) busca `alpha` en `np.logspace(-2, 4, 40)` para `make_pipeline(StandardScaler(), Ridge())` y para `make_pipeline(StandardScaler(), Lasso(max_iter=50_000))`. Los parámetros se llaman `ridge__alpha` y `lasso__alpha`.
3. Reporta el mejor `alpha` de cada uno y sus métricas en prueba. Agrégalos a `resultados_reg`.
4. Igual que en el cuaderno 01 (Lasso con `alpha` = 0.1, 1 y 10), estudia el efecto de `alpha`, pero ahora sobre los **coeficientes**: para cada `alpha` en `np.logspace(0, 4, 40)` ajusta el pipeline con Lasso, guarda los coeficientes y grafícalos contra `alpha` (eje x logarítmico).
5. **Preguntas:**
   - ¿Mejoró la regularización el desempeño en prueba? ¿Por qué tiene sentido con ≈ 1 070 filas y 11 variables?
   - ¿Qué variables lleva Lasso a cero primero al aumentar `alpha`? ¿Coincide con los p-valores del Ejercicio 2?
   - ¿Por qué hay que escalar antes de Ridge/Lasso?

In [ ]:
# Escribe tu código aquí
cv_reg = KFold(n_splits=5, shuffle=True, random_state=SEED)
alphas = np.logspace(-2, 4, 40)

---

<a id="ejercicio-4"></a>
<h2 style="color: #007ACC;">Ejercicio 4. KNN, SVR y árbol de regresión</h2>

**Objetivo:** ajustar modelos no lineales con búsqueda de hiperparámetros y compararlos con los lineales (cuadernos 06, 07, 08 y 10).

Usa las **variables originales** (`X_tr_r`, sin las transformaciones del Ejercicio 2) para ver si estos modelos descubren la estructura por sí solos.
Usa `GridSearchCV` con `cv=cv_reg` y `scoring="neg_mean_absolute_error"` en todos los casos.

1. **KNN:** `make_pipeline(StandardScaler(), KNeighborsRegressor())` con `n_neighbors` en `range(1, 41, 2)` y `metric` en `["euclidean", "manhattan"]`.
2. **SVR:** `make_pipeline(StandardScaler(), SVR())`. Recuerda (cuaderno 06) que $\epsilon$ es el ancho del "tubo" alrededor de la predicción y se mide **en las unidades de $y$** (aquí, dólares). Busca en una lista de dos grillas:
   - kernel `linear` con `C` en `[100, 1_000, 10_000]` y `epsilon` en `[100, 1_000]`;
   - kernel `rbf` con `C` en `[1_000, 10_000, 100_000]`, `gamma` en `["scale", 0.1, 1]` y `epsilon` en `[100, 1_000]`.
3. **Árbol:** `DecisionTreeRegressor(random_state=SEED)` con `max_depth` en `[2, 3, 4, 5, 6, 8, 10, None]` y `min_samples_leaf` en `[1, 5, 10, 20, 40]`. ¿Hace falta escalar para un árbol? (Revisa las ventajas del árbol en el cuaderno 08).
4. Para cada modelo reporta los mejores hiperparámetros, el MAE de entrenamiento y las métricas de prueba. Completa `resultados_reg` y grafica el MAE de prueba de todos los modelos en un gráfico de barras.
5. **Preguntas:**
   - ¿Qué modelo gana? ¿Por qué el árbol, sin la variable `fumador_obeso`, se acerca al modelo lineal que sí la tiene?
   - Ajusta aparte un árbol **sin límite de profundidad** y compara su MAE de entrenamiento y de prueba. Relaciónalo con el **sobreentrenamiento** (cuaderno 00).
   - ¿Qué ocurre con KNN cuando $k=1$? ¿Y cuando $k$ es muy grande?

In [ ]:
# Escribe tu código aquí

---

<a id="ejercicio-5"></a>
<h2 style="color: #007ACC;">Ejercicio 5. Interpretación de modelos de regresión</h2>

**Objetivo:** explicar *qué* aprendió cada modelo y verificar si cuentan la misma historia.

1. **Reglas del árbol.** Ajusta un `DecisionTreeRegressor(max_depth=3)` (para que sea legible) y dibújalo con `plot_tree` (cuaderno 08). Traduce las tres hojas con mayor costo predicho a reglas en español ("si fuma y ...").
2. **¿Qué variables importan?** Reúne la evidencia que ya tienes:
   - signo, magnitud y p-valor de los coeficientes del modelo lineal (Ejercicio 2);
   - el orden en que Lasso anula los coeficientes (Ejercicio 3);
   - las variables que usa el árbol en sus primeras divisiones.

   ¿Coinciden?
3. **Curvas "qué pasaría si".** Como en los cuadernos 07 y 08 (donde graficamos la predicción sobre una grilla de valores de $x$), construye una grilla de IMC entre 18 y 50 para una persona de 45 años, hombre, 2 hijos, región `southeast`. Grafica la predicción de cada modelo (lineal, lineal con transformaciones, KNN, SVR y árbol) en función del IMC, con una curva para **fumador** y otra para **no fumador**.
   - Pista: escribe una función `crear_personas(...)` que arme el DataFrame con las mismas columnas de `X_reg`.
4. Para la misma persona con IMC 33, construye una tabla con la predicción de cada modelo si fuma y si no fuma, y la diferencia. ¿Cuánto "cuesta" fumar según cada modelo? ¿Por qué el modelo lineal sin transformaciones da una cifra distinta a los demás?

In [ ]:
# Escribe tu código aquí

---

<h1 style="color: #FECB05;">Parte II — Clasificación: diabetes en mujeres pima</h1>

El *National Institute of Diabetes and Digestive and Kidney Diseases* siguió a mujeres de la comunidad pima (Arizona) de 21 años o más.
Queremos predecir si una paciente **desarrollará diabetes en los próximos 5 años** para priorizar un programa de prevención.

| Variable | Descripción |
|---|---|
| `embarazos` | Número de embarazos |
| `glucosa` | Glucosa plasmática a las 2 h en prueba oral de tolerancia (mg/dL) |
| `presion` | Presión diastólica (mm Hg) |
| `pliegue` | Pliegue cutáneo del tríceps (mm) |
| `insulina` | Insulina sérica a las 2 h (μU/mL) |
| `imc` | Índice de masa corporal |
| `pedigri` | Función de pedigrí de diabetes (antecedentes familiares) |
| `edad` | Años |
| `diabetes` | **Objetivo**: 1 = desarrolló diabetes |

El archivo **no trae encabezado**; usa la lista `cols_pima` de abajo.

In [ ]:
cols_pima = ["embarazos", "glucosa", "presion", "pliegue", "insulina", "imc", "pedigri", "edad", "diabetes"]

---

<a id="ejercicio-6"></a>
<h2 style="color: #007ACC;">Ejercicio 6. Exploración y limpieza</h2>

**Objetivo:** detectar problemas de calidad que no se ven con `isna()` y preparar una partición estratificada.

1. Carga `URL_PIMA` con `header=None, names=cols_pima` en `pima`. Verifica `isna().sum()`.
2. Cuenta cuántos **ceros** hay en cada variable. ¿Cuáles de esos ceros son **fisiológicamente imposibles**? (Nadie vivo tiene glucosa, presión o IMC igual a 0).
3. Limpieza:
   - elimina las columnas `insulina` y `pliegue` (demasiados ceros imposibles para usarlas);
   - elimina las filas con cero en `glucosa`, `presion` o `imc`. ¿Cuántas filas quedan?
4. Muestra la proporción de cada clase, los boxplots de cada variable por clase y el mapa de calor de correlaciones.
5. Separa `X_clf` y `y_clf` y haz una partición 75/25 **estratificada** (`stratify=y_clf`, cuaderno 10) con `random_state=SEED`: `X_tr_c, X_te_c, y_tr_c, y_te_c`.
6. **Referencia.** ¿Qué exactitud y qué sensibilidad tendría en prueba un "modelo" que siempre responde 0 (no diabetes)? Calcúlalo con `metricas_clasificacion`.
7. **Preguntas:**
   - ¿Qué pasaría si dejáramos los ceros de `insulina` como valores reales en un modelo lineal o en KNN?
   - ¿Por qué estratificar? ¿Por qué la exactitud de la referencia es engañosa aquí?

In [ ]:
# Escribe tu código aquí
pima = pd.read_csv(URL_PIMA, header=None, names=cols_pima)

---

<a id="ejercicio-7"></a>
<h2 style="color: #007ACC;">Ejercicio 7. Regresión logística y matriz de confusión</h2>

**Objetivo:** entrenar el clasificador lineal de referencia, evaluarlo con las métricas de la matriz de confusión e interpretar sus coeficientes (cuadernos 04 y 05).

1. Entrena `logit = make_pipeline(StandardScaler(), LogisticRegression())`.
2. En prueba: dibuja la matriz de confusión (`ConfusionMatrixDisplay`), imprime `classification_report` y agrega `metricas_clasificacion` a `resultados_clf`.
3. A partir de la matriz (`confusion_matrix`), identifica VP, FP, VN y FN (cuaderno 05) y calcula **a mano** la exactitud, la precisión, la sensibilidad y la F1. Verifica que coincidan con sklearn (es la tarea que quedó propuesta en el cuaderno 04).
4. **Coeficientes.** El modelo calcula $\hat{p} = \sigma(\mathbf{x}^T\boldsymbol{\beta})$. Como las variables están estandarizadas, cada $\beta_j$ indica cuánto cambia $\mathbf{x}^T\boldsymbol{\beta}$ cuando la variable aumenta **una desviación estándar**. Grafica los coeficientes ordenados.
   - ¿Qué signo tienen? Un $\beta_j > 0$, ¿aumenta o disminuye $\hat{p}$?
   - ¿Por qué podemos comparar sus magnitudes entre variables solo porque escalamos?
5. **Pregunta:** en este contexto, ¿qué error de la matriz de confusión es más grave: el falso positivo o el falso negativo? ¿Qué métrica priorizarías?

In [ ]:
# Escribe tu código aquí

---

<a id="ejercicio-8"></a>
<h2 style="color: #007ACC;">Ejercicio 8. El umbral de decisión</h2>

**Objetivo:** entender que la regresión logística produce una probabilidad $\hat{p}$ y que la clase se decide con un umbral.

En el cuaderno 04 definimos $\hat{y} = 1$ si $\hat{p} \geq 0.5$. Ese 0.5 es una **decisión**, no algo que el modelo aprenda. Veamos qué pasa si lo movemos.

1. Con `logit.predict_proba(X)[:, 1]` se obtiene $\hat{p}$ para cada paciente. Comprueba que `(p_hat >= 0.5)` reproduce exactamente `logit.predict(X_te_c)`.
2. **Elegir el umbral sin mirar la prueba.** Usa el mismo esquema de K-Folds del cuaderno 10 (`KFold(n_splits=5, shuffle=True, random_state=SEED)` y un ciclo `for` sobre `kf.split(X_tr_c)`): en cada partición entrena la logística con $K-1$ subconjuntos y calcula $\hat{p}$ del subconjunto que quedó por fuera. Al final tendrás un $\hat{p}$ "fuera de muestra" para cada paciente de entrenamiento.
3. Para umbrales en `np.arange(0.05, 0.96, 0.05)` calcula la precisión, la sensibilidad y la F1 con esas probabilidades y grafícalas contra el umbral.
4. Elige el umbral **más alto** que garantice una sensibilidad ≥ 0.80.
5. Aplica los umbrales 0.5 y el elegido **una sola vez** al conjunto de prueba y compara matrices de confusión y métricas.
6. **Preguntas:**
   - ¿Por qué no elegir el umbral directamente con el conjunto de prueba?
   - ¿Qué umbral recomendarías para un programa de tamizaje barato? ¿Y para decidir un tratamiento costoso con efectos adversos?

In [ ]:
# Escribe tu código aquí
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

---

<a id="ejercicio-9"></a>
<h2 style="color: #007ACC;">Ejercicio 9. KNN, SVM y árbol de clasificación</h2>

**Objetivo:** ajustar los tres clasificadores no lineales, con atención a lo que controla cada hiperparámetro (cuadernos 06, 07, 08 y 10).

Define `cv_clf = KFold(n_splits=5, shuffle=True, random_state=SEED)` y usa `scoring="f1"` en todas las búsquedas (en el cuaderno 04 vimos que la F1 combina precisión y sensibilidad, útil cuando la exactitud engaña).

**9a. KNN**

1. Con `cross_val_score` compara la F1 de un KNN con `k=15` **sin escalar** vs. **con** `StandardScaler`. Explica la diferencia mirando las escalas de `glucosa` y `pedigri` (`X_tr_c.describe()`).
2. Igual que en el cuaderno 10, recorre $k$ en `range(1, 80, 2)` y grafica la F1 de validación cruzada contra $k$. Agrega la F1 en **entrenamiento** (ajustando y prediciendo sobre `X_tr_c`). Señala las zonas de sobreentrenamiento y subentrenamiento.
3. Con `GridSearchCV` busca `n_neighbors` en `range(1, 80, 2)` y `metric` en `["euclidean", "manhattan"]` (cuaderno 07).

**9b. SVM**

4. Busca con `make_pipeline(StandardScaler(), SVC())` en una lista de grillas: kernel `linear` con `C` en `[0.01, 0.1, 1, 10]`; kernel `rbf` con `C` en `[0.1, 1, 10, 100]` y `gamma` en `[0.001, 0.01, 0.1, 1]`; kernel `poly` con `degree` en `[2, 3]` y `C` en `[0.1, 1, 10]`.
5. Con el mejor modelo reporta cuántos **vectores de soporte** tiene (`n_support_`) y qué fracción del entrenamiento representan. ¿Qué dice eso sobre qué tan separables son las clases?
6. Con `cv_results_` arma un mapa de calor de la F1 de validación para el kernel `rbf` (`C` × `gamma`). ¿Qué ocurre con `gamma=1`?

**9c. Árbol**

7. Para `max_depth` en `1..15` grafica la F1 de entrenamiento y la de validación cruzada. ¿Desde qué profundidad sobreentrena?
8. Busca `criterion` en `["gini", "entropy"]` (cuaderno 08), `max_depth` en `[2, 3, 4, 5, 6, 8]` y `min_samples_leaf` en `[1, 5, 10, 20, 40]`. ¿Importa el criterio?

In [ ]:
# Escribe tu código aquí
cv_clf = KFold(n_splits=5, shuffle=True, random_state=SEED)

# 9a. KNN

In [ ]:
# Escribe tu código aquí
# 9b. SVM

In [ ]:
# Escribe tu código aquí
# 9c. Árbol

---

<a id="ejercicio-10"></a>
<h2 style="color: #007ACC;">Ejercicio 10. Comparación con validación cruzada</h2>

**Objetivo:** comparar los modelos de forma justa (mismas particiones, varias métricas, con su variabilidad) y solo al final mirar la prueba.

1. Reúne en un diccionario `modelos_clf` los mejores estimadores: Logística, KNN, SVM y Árbol.
2. Con `cross_val_score` y `cv_clf`, calcula para cada modelo la media y la desviación estándar de `"accuracy"`, `"precision"`, `"recall"` y `"f1"` sobre **entrenamiento**, y arma una tabla.
3. ¿Las diferencias de F1 entre modelos son mayores que la desviación estándar entre particiones?
4. Ahora sí, evalúa en **prueba**: dibuja las cuatro matrices de confusión lado a lado y agrega sus métricas a `resultados_clf`.
5. **Pregunta:** si tuvieras que elegir un modelo, ¿cuál y por qué? Considera desempeño, estabilidad e interpretabilidad (revisa las ventajas y desventajas de cada modelo en los cuadernos).

In [ ]:
# Escribe tu código aquí

---

<a id="ejercicio-11"></a>
<h2 style="color: #007ACC;">Ejercicio 11. Interpretación de clasificadores</h2>

**Objetivo:** explicar el modelo de forma **global** (qué variables importan y cómo es la frontera de decisión) y **local** (por qué esta paciente recibió esta predicción).

1. **Reglas del árbol.** Ajusta un `DecisionTreeClassifier(max_depth=3, min_samples_leaf=20)` y dibújalo con `plot_tree` (`class_names=["No", "Diabetes"]`, `proportion=True`). Redacta la regla que identifica el grupo de **mayor riesgo** y la de **menor riesgo**, con la proporción de diabetes de cada hoja. ¿Qué variable usa en la raíz? ¿Coincide con el coeficiente más grande de la logística?
2. **Fronteras de decisión.** Como en los cuadernos 06, 07 y 08, entrena los cuatro modelos usando solo `glucosa` e `imc` (con los mejores hiperparámetros encontrados) y dibuja sus fronteras con `DecisionBoundaryDisplay` sobre los datos de entrenamiento. Describe la forma de cada frontera.
3. **Una paciente.** Toma, de las pacientes **con diabetes** del conjunto de prueba, la que tenga la **menor $\hat{p}$** según la logística (el falso negativo "más seguro" del modelo).
   - Calcula **a mano** $t = \beta_0 + \sum_j \beta_j z_j$ (con $z_j$ la variable estandarizada) y $\hat{p} = \sigma(t) = 1/(1 + e^{-t})$. Verifica que coincide con `predict_proba`.
   - Grafica las contribuciones $\beta_j z_j$ de cada variable. ¿Qué la hace parecer de bajo riesgo?
   - ¿Qué predicen KNN y el árbol para ella? Recuerda que KNN asigna la clase de la mayoría de sus vecinos.
4. **Pregunta:** `edad` y `embarazos` están correlacionadas (revisa el mapa de calor). ¿Qué problema genera la **multicolinealidad** (cuaderno 03) al interpretar sus coeficientes?

In [ ]:
# Escribe tu código aquí

---

<a id="ejercicio-integrador"></a>
<h2 style="color: #007ACC;">Ejercicio integrador. Informe para el comité</h2>

Escribe, **en no más de una página** (Markdown en este cuaderno), dos informes breves dirigidos a personas no técnicas:

**A. Aseguradora (Parte I)**

- Modelo recomendado para tarificar y su error esperado (en USD, con la métrica adecuada).
- Los tres factores que más mueven el costo y la magnitud de su efecto, incluido el efecto conjunto de fumar y la obesidad.
- Un riesgo ético o regulatorio de usar `sex` o `region` en la prima, y si el modelo realmente las necesita (usa tu evidencia).

**B. Programa de prevención de diabetes (Parte II)**

- Modelo y **umbral** recomendados, con la matriz de confusión esperada por cada 100 pacientes evaluadas.
- Qué significan en la práctica la sensibilidad y la precisión elegidas (cuántas pacientes en riesgo se escapan y cuántas citas "innecesarias" se generan).
- Dos limitaciones del análisis (por ejemplo: población específica, tamaño de muestra, variables eliminadas, asociación vs. causalidad) y qué datos adicionales pedirías.

*Escribe tu informe aquí.*